# QPU Benchmark Analysis – QFT & GHZ

Analyses results from `submit_QFT.py` and `submit_GHZ.py` runs across AWS QPU devices.

**Sections**
- **Section 0** – Imports and helper functions
- **Section 1** – Local simulator baseline (QFT and GHZ, no noise)
- **Section 2** – Hardware results, per device:
  - 2-1 IonQ Forte-1
  - 2-2 IonQ Forte-Enterprise-1
  - 2-3 Rigetti Ankaa-3
  - 2-4 IQM Garnet
- **Section 3** – Cross-device comparison

**Metrics**
- *QFT*: Total Variation Distance (TVD) from ideal uniform output distribution
- *GHZ*: Ideal fraction — fraction of shots in |00…0⟩ or |11…1⟩
- *Timing*: queue time, wall time, QPU execution time (if reported by device)

---
## Section 0 – Imports and Helper Functions

In [ ]:
import json
import os
import glob
import math

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from braket.circuits import Circuit
from braket.devices import LocalSimulator

JOB_RESULTS_DIR = os.path.join(os.getcwd(), 'job_results')
LOG_FILE        = os.path.join(JOB_RESULTS_DIR, 'job_log.txt')

print('job_results dir:', JOB_RESULTS_DIR)

In [ ]:

# ── Circuit builders ──────────────────────────────────────────────────────────

def build_qft(n: int) -> Circuit:
    """n-qubit QFT (H, CPhaseShift, SWAP)."""
    circ = Circuit()
    for i in range(n):
        circ.h(i)
        for j in range(i + 1, n):
            circ.cphaseshift(j, i, np.pi / 2 ** (j - i))
    for i in range(n // 2):
        circ.swap(i, n - 1 - i)
    return circ


def build_ghz(n: int) -> Circuit:
    """n-qubit GHZ state: H on qubit 0, then CNOT chain."""
    circ = Circuit()
    circ.h(0)
    for i in range(n - 1):
        circ.cnot(i, i + 1)
    return circ


# ── Metrics ───────────────────────────────────────────────────────────────────

def tvd_from_uniform(counts: dict) -> float:
    """
    Total Variation Distance of measured distribution from uniform.
    QFT on |0...0⟩ ideally produces uniform over all 2^n bitstrings.
    Lower is better (0 = perfect).
    """
    total    = sum(counts.values())
    n_states = 2 ** len(next(iter(counts))) if counts else 1
    p_ideal  = 1.0 / n_states
    tvd      = 0.5 * sum(abs(cnt / total - p_ideal) for cnt in counts.values())
    n_unobs  = n_states - len(counts)
    tvd     += 0.5 * n_unobs * p_ideal
    return tvd


def ghz_ideal_fraction(counts: dict, n_qubits: int) -> float:
    """Fraction of shots in |00...0⟩ or |11...1⟩."""
    ideal       = {'0' * n_qubits, '1' * n_qubits}
    total       = sum(counts.values())
    ideal_shots = sum(cnt for k, cnt in counts.items() if k in ideal)
    return ideal_shots / total if total else 0.0


# ── Plotting helpers ──────────────────────────────────────────────────────────

def plot_histogram(counts: dict, title: str, top_k: int = 20, ax=None):
    """Bar chart of the top_k most frequent outcomes. X-axis shows integer values."""
    total  = sum(counts.values())
    top    = sorted(counts.items(), key=lambda x: -x[1])[:top_k]
    labels = [int(k, 2) for k, _ in top]   # bitstring → integer
    vals   = [v / total for _, v in top]

    if ax is None:
        fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.5), 4))

    ax.bar(range(len(labels)), vals, color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, fontsize=8)
    ax.set_xlabel('Outcome (decimal)')
    ax.set_ylabel('Probability')
    ax.set_title(title)
    return ax


def print_timing(result: dict):
    """Print timing summary for a retrieved result."""
    q  = result.get('queue_time_seconds')
    w  = result.get('wall_time_seconds')
    qp = result.get('qpu_time_seconds')
    print(f"  Queue+exec time : {q:.1f} s"  if q  is not None else "  Queue+exec time : N/A")
    print(f"  Wall time       : {w:.1f} s"  if w  is not None else "  Wall time       : N/A")
    print(f"  QPU exec time   : {qp:.4f} s" if qp is not None else "  QPU exec time   : not reported")


# ── Result loader ─────────────────────────────────────────────────────────────

def load_results(device_name: str | None = None,
                 circuit_type: str | None = None) -> list[dict]:
    """Load retrieved result JSONs from job_results/, with optional filters."""
    results = []
    for path in sorted(glob.glob(os.path.join(JOB_RESULTS_DIR, '*.json'))):
        with open(path) as f:
            rec = json.load(f)
        if 'circuit_type' not in rec:
            continue
        if device_name  and rec.get('device')       != device_name:
            continue
        if circuit_type and rec.get('circuit_type') != circuit_type:
            continue
        results.append(rec)
    return results


print('Helpers loaded.')


---
## Section 1 – Local Simulator Baseline

Runs QFT and GHZ circuits on the Braket `LocalSimulator` (no noise).  
These are the ideal reference results.

**Ideal outputs**
- QFT applied to |0...0⟩ → uniform distribution over all 2ⁿ bitstrings  
- GHZ state → 50 % |00...0⟩ and 50 % |11...1⟩

In [ ]:
SIM_N_QUBITS = 6    # qubit count for local sim demo
SIM_SHOTS    = 1024  # shots for local sim

simulator = LocalSimulator()
print(f'LocalSimulator ready. n_qubits={SIM_N_QUBITS}, shots={SIM_SHOTS}')

### 1-1: QFT – Circuit Display and Local Simulation

In [ ]:
qft_circ = build_qft(SIM_N_QUBITS)
print(f'QFT circuit  n={SIM_N_QUBITS}  depth={qft_circ.depth}  gates={len(qft_circ.instructions)}')
print()
print(qft_circ.diagram())

In [ ]:
qft_task   = simulator.run(qft_circ, shots=SIM_SHOTS)
qft_result = qft_task.result()
qft_counts = dict(qft_result.measurement_counts)

qft_tvd = tvd_from_uniform(qft_counts)

print(f'QFT local simulation  (n={SIM_N_QUBITS}, shots={SIM_SHOTS})')
print(f'  Unique outcomes : {len(qft_counts)}')
print(f'  TVD from uniform: {qft_tvd:.4f}  (0 = ideal uniform)')
print(f'  Top 5 outcomes  :')
for k, v in sorted(qft_counts.items(), key=lambda x: -x[1])[:5]:
    print(f'    {k}  →  {v} shots  ({v/SIM_SHOTS:.3f})')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_histogram(qft_counts,
               title=f'QFT |0^{SIM_N_QUBITS}⟩ — Local Simulator  '
                     f'(n={SIM_N_QUBITS}, shots={SIM_SHOTS}, TVD={qft_tvd:.4f})',
               top_k=32, ax=ax)
# Draw ideal uniform line
ax.axhline(1 / 2**SIM_N_QUBITS, color='red', linestyle='--', label='Ideal uniform')
ax.legend()
plt.tight_layout()
plt.show()

### 1-2: GHZ – Circuit Display and Local Simulation

In [ ]:
ghz_circ = build_ghz(SIM_N_QUBITS)
print(f'GHZ circuit  n={SIM_N_QUBITS}  depth={ghz_circ.depth}  gates={len(ghz_circ.instructions)}')
print()
print(ghz_circ.diagram())

In [ ]:
ghz_task   = simulator.run(ghz_circ, shots=SIM_SHOTS)
ghz_result = ghz_task.result()
ghz_counts = dict(ghz_result.measurement_counts)

ghz_frac = ghz_ideal_fraction(ghz_counts, SIM_N_QUBITS)

print(f'GHZ local simulation  (n={SIM_N_QUBITS}, shots={SIM_SHOTS})')
print(f'  Unique outcomes   : {len(ghz_counts)}')
print(f'  Ideal fraction    : {ghz_frac:.4f}  (1.0 = perfect GHZ)')
print(f'  All outcomes      :')
for k, v in sorted(ghz_counts.items(), key=lambda x: -x[1]):
    label = '  ← ideal' if k in {'0'*SIM_N_QUBITS, '1'*SIM_N_QUBITS} else ''
    print(f'    {k}  →  {v} shots  ({v/SIM_SHOTS:.3f}){label}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_histogram(ghz_counts,
               title=f'GHZ — Local Simulator  '
                     f'(n={SIM_N_QUBITS}, shots={SIM_SHOTS}, ideal_frac={ghz_frac:.4f})',
               top_k=len(ghz_counts), ax=ax)
# Mark ideal states
keys = sorted(ghz_counts.items(), key=lambda x: -x[1])
ideal_set = {'0'*SIM_N_QUBITS, '1'*SIM_N_QUBITS}
for idx, (k, _) in enumerate(keys):
    if k in ideal_set:
        ax.get_children()[idx].set_color('orange')
ax.axhline(0.5, color='red', linestyle='--', label='Ideal 50 %')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 2 – Hardware Results

Run `checkRetieve.py` on the DCV first to populate `job_results/*.json`.  
Each sub-section loads results for one QPU device.

For each device two jobs are expected per circuit type:
- **half**: n = max_qubits // 2
- **full**: n = max_qubits

| # | Device | Company | Qubits | Notes |
|---|--------|---------|--------|-------|
| 2-1 | Forte-1 | IonQ | 36 | trapped-ion |
| 2-2 | Forte-Enterprise-1 | IonQ | 36 | trapped-ion |
| 2-3 | Ankaa-3 | Rigetti | 84 | superconducting |
| 2-4 | Garnet | IQM | 20 | superconducting |
| 2-5 | Ibex-Q1 | AQT | 12 | trapped-ion, eu-north-1 |
| 2-6 | Emerald | IQM | TBD | superconducting, eu-north-1 |
| 2-7 | Cepheus-1-108Q | Rigetti | 108 | superconducting |

In [ ]:
# Load all available hardware results
all_results = load_results()

if not all_results:
    print('No hardware results found yet in', JOB_RESULTS_DIR)
    print('Run checkRetieve.py on the DCV first.')
else:
    print(f'Loaded {len(all_results)} hardware result(s):')
    print(f'  {"Device":24s}  {"Circuit":6s}  {"Qubits":6s}  {"Shots":6s}  {"Status"}')
    print(f'  {"-"*65}')
    for r in all_results:
        uuid = r['task_id'].split('/')[-1][:8]
        print(f'  {r["device"]:24s}  {r["circuit_type"]:6s}  '
              f'{r["n_qubits"]:6d}  {r["n_shots_actual"]:6d}  {uuid}...')

### Per-device analysis helper

In [ ]:
def analyse_device(device_name: str):
    """
    Load and display all results for a given device.
    Shows histograms, metrics, and timing for each (circuit_type, n_qubits) pair.
    """
    results = load_results(device_name=device_name)
    if not results:
        print(f'  No results found for {device_name}.')
        return

    print(f'\n{"="*70}')
    print(f'  Device : {device_name}  ({len(results)} result(s))')

    # Sort by circuit type then n_qubits
    results.sort(key=lambda r: (r['circuit_type'], r['n_qubits']))

    for r in results:
        ctype   = r['circuit_type']
        n       = r['n_qubits']
        shots   = r['n_shots_actual']
        counts  = r['counts']
        uuid    = r['task_id'].split('/')[-1]
        label   = 'half' if r.get('n_qubits') == r.get('n_qubits') else 'full'  # placeholder

        print(f'\n  [{ctype}  n={n}]  shots={shots}  task={uuid[:8]}...')

        # Metric
        if ctype == 'QFT':
            tvd = tvd_from_uniform(counts)
            print(f'    TVD from uniform : {tvd:.4f}  (lower = closer to ideal)')
        elif ctype == 'GHZ':
            frac = ghz_ideal_fraction(counts, n)
            print(f'    GHZ ideal frac   : {frac:.4f}  (1.0 = perfect)')
            if 'ghz_verification' in r:
                v = r['ghz_verification']
                print(f'    Ideal shots      : {v["ideal_shots"]}/{v["total_shots"]}')

        # Timing
        print_timing(r)

        # Histogram
        fig, ax = plt.subplots(figsize=(12, 3))
        plot_histogram(
            counts,
            title=f'{device_name}  [{ctype}  n={n}]  shots={shots}',
            top_k=20,
            ax=ax,
        )
        if ctype == 'QFT':
            ax.axhline(1 / 2**n, color='red', linestyle='--', label='Ideal uniform')
            ax.legend()
        elif ctype == 'GHZ':
            ax.set_title(ax.get_title() + f'  |  ideal_frac={ghz_ideal_fraction(counts, n):.4f}')
        plt.tight_layout()
        plt.show()


print('analyse_device() ready.')

---
### 2-1: IonQ Forte-1

In [ ]:
analyse_device('Forte-1')

---
### 2-2: IonQ Forte-Enterprise-1

In [ ]:
analyse_device('Forte-Enterprise-1')

---
### 2-3: Rigetti Ankaa-3

In [ ]:
analyse_device('Ankaa-3')

---
### 2-4: IQM Garnet

In [ ]:
analyse_device('Garnet')

---
### 2-5: AQT Ibex-Q1

In [ ]:
analyse_device('Ibex-Q1')

---
### 2-6: IQM Emerald

In [ ]:
analyse_device('Emerald')

---
### 2-7: Rigetti Cepheus-1-108Q

In [ ]:
analyse_device('Cepheus-1-108Q')

---
## Section 3 – Cross-Device Comparison

Bar charts comparing each metric across all available devices.

In [ ]:
# ── QFT: TVD from uniform ─────────────────────────────────────────────────────
qft_results = load_results(circuit_type='QFT')

if not qft_results:
    print('No QFT results yet.')
else:
    labels = []
    tvds   = []
    for r in sorted(qft_results, key=lambda x: (x['device'], x['n_qubits'])):
        labels.append(f"{r['device']}\nn={r['n_qubits']}")
        tvds.append(tvd_from_uniform(r['counts']))

    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.2), 4))
    bars = ax.bar(range(len(labels)), tvds, color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('TVD from uniform  (lower = better)')
    ax.set_title('QFT Benchmark – TVD from Uniform Distribution (per device & qubit count)')
    for bar, v in zip(bars, tvds):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── GHZ: Ideal fraction ───────────────────────────────────────────────────────
ghz_results = load_results(circuit_type='GHZ')

if not ghz_results:
    print('No GHZ results yet.')
else:
    labels = []
    fracs  = []
    for r in sorted(ghz_results, key=lambda x: (x['device'], x['n_qubits'])):
        labels.append(f"{r['device']}\nn={r['n_qubits']}")
        fracs.append(ghz_ideal_fraction(r['counts'], r['n_qubits']))

    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.2), 4))
    bars = ax.bar(range(len(labels)), fracs, color='darkorange', alpha=0.8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.axhline(1.0, color='red', linestyle='--', label='Ideal (1.0)')
    ax.set_ylabel('Ideal fraction  (higher = better)')
    ax.set_title('GHZ Benchmark – Ideal Fraction |00…0⟩ + |11…1⟩ (per device & qubit count)')
    ax.legend()
    for bar, v in zip(bars, fracs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Timing comparison ─────────────────────────────────────────────────────────
if not all_results:
    print('No results to plot timing for.')
else:
    timing_data = []
    for r in all_results:
        wall  = r.get('wall_time_seconds')
        queue = r.get('queue_time_seconds')
        qpu   = r.get('qpu_time_seconds')
        if wall is not None or queue is not None:
            timing_data.append({
                'label':       f"{r['device']}\n{r['circuit_type']} n={r['n_qubits']}",
                'wall':        wall,
                'queue':       queue,
                'qpu':         qpu,
            })

    if not timing_data:
        print('Timing data not yet available in results.')
    else:
        fig, ax = plt.subplots(figsize=(max(8, len(timing_data) * 1.2), 4))
        x      = range(len(timing_data))
        labels = [d['label'] for d in timing_data]
        walls  = [d['wall']  or 0 for d in timing_data]
        queues = [d['queue'] or 0 for d in timing_data]

        ax.bar(x, walls,  label='Wall time (s)',       alpha=0.7, color='steelblue')
        ax.bar(x, queues, label='Queue+exec time (s)', alpha=0.7, color='darkorange', width=0.4)

        # Overlay QPU time if available
        qpu_times = [d['qpu'] for d in timing_data]
        if any(q is not None for q in qpu_times):
            qpu_y = [q or 0 for q in qpu_times]
            ax.scatter(x, qpu_y, color='red', zorder=5, label='QPU exec time (s)', s=60)

        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylabel('Time (seconds)')
        ax.set_title('QPU Task Timing')
        ax.legend()
        plt.tight_layout()
        plt.show()

        print('Timing summary:')
        print(f'  {"Label":40s}  {"Wall (s)":10s}  {"Queue (s)":10s}  {"QPU (s)"}')
        print(f'  {"-"*80}')
        for d in timing_data:
            lbl   = d['label'].replace('\n', ' ')
            wall  = f"{d['wall']:.1f}"  if d['wall']  is not None else 'N/A'
            queue = f"{d['queue']:.1f}" if d['queue'] is not None else 'N/A'
            qpu   = f"{d['qpu']:.4f}"  if d['qpu']   is not None else 'N/A'
            print(f'  {lbl:40s}  {wall:10s}  {queue:10s}  {qpu}')